# iDeepLBCE - PSTPP held-out evaluation on the Beetle dataset
This notebook evaluates classical machine learning and deep learning models using only PSTPP features extracted from sequences prepared following the guidance in the Beetle article.

Set DATA_ROOT in Settings to the directory containing these three folders:
| Folder | Contents | Expected samples |
|---|---|---:|
| `train_beetle` | PSTPP features for the training set | 119,503 |
| `val_beetle` | PSTPP features for the validation set | 10,000 |
| `test_beetle` | PSTPP features for the held-out test set | 10,000 |

Each folder must contain exactly one CSV whose filename begins with PSTPP (case-insensitive), such as PSTPP_train_beetle.csv. Use CSV_OVERRIDES if a file requires an explicit path.

All CSV files must be without headers. The first column contains the class label (0 = non-epitope, 1 = epitope), followed by 23 numerical PSTPP features. Files are read using header=None to retain the first sample, and feature column names are generated automatically. Preserve the original sequence order when extracting features and assigning labels.

Only PSTPP is selected in ML_JOBS and DL_JOBS. The current configuration evaluates:
1. Classical models: XGBoost (XGB), Random Forest (RF), Logistic Regression (LR), and Extra Trees (ET).
2. Deep learning models: CNN, FCNN, RNN, LSTM, and GRU.

The F1 and F2 feature groups remain defined for compatibility but are not used in these experiments.

After configuring the paths, run the shared cells followed by either or both model sections. Use a Python runtime with pandas, scikit-learn, XGBoost, and TensorFlow with Keras 3. A GPU can accelerate deep learning training. In Colab, mount Google Drive first if the files are stored there.

The current configuration uses seed 42 and a fixed classification threshold of 0.50. Deep learning models train on train_beetle, use val_beetle for validation and callbacks, and allow up to 200 epochs with batch size 32. Early stopping monitors validation loss with patience 50 and restores the best weights. Both validation and test metrics are reported; reserve the held-out test set for final evaluation.

Results are saved in a new timestamped directory under _results_beetle. Supply multiple seeds to

In [16]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass
import copy
import csv
import json
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, matthews_corrcoef,
    roc_auc_score,
)

### Settings

Change the path in `DATA_ROOT` variable accordingly.

To use the notebook in `Kaggle` (example): /kaggle/input/datasets/name/features

To use the notebook in `Colab` (example): /content/drive/MyDrive/features

In [28]:
# ROOT = "ml_dl_pstpp_Beetle_evaluation" 
# print('Root directory:', ROOT)
DATA_ROOT = Path('./features_beetle')  
DATA_DIRS = {name: DATA_ROOT / name for name in (
    'train_beetle', 'val_beetle', 'test_beetle'
)}
# Optional per-file overrides: {('int_ibce', 'PSSM'): Path('/path/to/file.csv')}
CSV_OVERRIDES = {}

MODEL_JSON_DIR = None
# OUTPUT_ROOT = Path('/kaggle/working' if Path('/kaggle/working').is_dir() else '/content') / '_results_beetle'
OUTPUT_ROOT = Path(".") / "_results_beetle"

ML_JOBS = {'PSTPP': ['XGB', 'RF', 'LR', 'ET']}
DL_JOBS = {'PSTPP': ['CNN', 'FCNN', 'RNN', 'LSTM', 'GRU']}

SEEDS = [42]  # Example repeated fits: [42, 43, 44]
THRESHOLD = 0.50
VAL_FRACTION = 0.20
MAX_EPOCHS = 200
BATCH_SIZE = 32
PATIENCE = 50
MONITOR = 'val_loss' # 'val_accuracy' #
MODE = 'min' # 'max'
N_JOBS = 2
EXPECTED_ROWS = dict(train_beetle=119503, val_beetle=10000, test_beetle=10000)#, ind_abcpred=1231)
# Set EXPECTED_ROWS = {} for intentionally different datasets.

if not SEEDS or len(SEEDS) != len(set(SEEDS)) or not 0 < THRESHOLD < 1:
    raise ValueError('Use unique seeds and a threshold between 0 and 1.')
RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Results directory:', RUN_DIR)

Results directory: _results_beetle\20261122T123752_965835Z


In [29]:
DIMENSIONS = dict(AAC=20, DPC=400, PAAC=25, APAAC=30, PCP=9, PSSM=62, PSTPP=23, ONEHOT=525)
## No GROUPS or PREFIXES Will be used for the beetle dataset, but they are kept here for compatibility with the original code structure.
GROUPS = {'F1': ('AAC','DPC','PAAC','APAAC','PCP'), 'F2': ('ONEHOT','PSSM','PSTPP')}
PREFIXES = {'ONEHOT': ('onehot','one-hot','one_hot','one hot','ohe')}
ID_COLUMNS = ('peptide_id','seq_id','sample_id','orig_idx','sequence')
TESTS = ('val_beetle','test_beetle')

@dataclass
class Dataset:
    X: pd.DataFrame
    y: np.ndarray


def find_csv(split, block):
    # print(f"Split: {split} block: {block}") # <-- Test
    override = CSV_OVERRIDES.get((split, block))
    if override is not None:
        return Path(override)
    folder = Path(DATA_DIRS[split])
    aliases = PREFIXES.get(block, (block.lower(),))
    # print(f"aliases: {aliases} folder: {folder}") # <-- Test
  
    candidates = sorted(p for p in folder.iterdir() if p.is_file() and
                        p.suffix.lower() == '.csv' and p.stem.lower().startswith(aliases))
    if len(candidates) != 1:
        raise ValueError(f'{split}/{block}: expected one CSV in {folder}, found {candidates}. Set CSV_OVERRIDES.')
    return candidates[0]

def read_block(path, block):

    frame = pd.read_csv(
        path,
        dtype=str,
        header=None,
        keep_default_na=False,
        encoding="utf-8-sig",
        engine="python",
    )  # pd.read_csv(path, dtype=str, keep_default_na=False)


    labels = pd.to_numeric(frame.iloc[:, 0], errors="raise")
    if set(labels.unique()) != {0, 1}:
        raise ValueError(f"{path}: column 1 must contain both binary labels, 0 and 1.")

    X = (
        frame.iloc[:, 1:]
        .apply(pd.to_numeric, errors="raise").astype("float32")
    )
    if X.shape[1] != DIMENSIONS[block] or not np.isfinite(X.to_numpy()).all():
        raise ValueError(
            f"{path}: expected {DIMENSIONS[block]} finite numeric features, got {X.shape[1]} columns."
        )
    return Dataset(X.add_prefix(f"{block}:"), labels.to_numpy(dtype=int))  # , ids)


def load_feature(feature_name):
    feature = re.sub(r"[^A-Z0-9]", "", feature_name.upper())
    blocks = GROUPS.get(feature, (feature,))
    # print(blocks) # <--
    if any(block not in DIMENSIONS for block in blocks):
        raise ValueError(
            f"Unknown feature {feature_name}. Choose from {list(DIMENSIONS) + list(GROUPS)}."
        )
    result = {}
    for split in ("train_beetle", *TESTS):
        datasets = [read_block(find_csv(split, block), block) for block in blocks]
        base = datasets[0]
        # ids = base.ids.copy()
        expected = EXPECTED_ROWS.get(split)
        if expected is not None and len(base.y) != expected:
            raise ValueError(
                f"{split}: found {len(base.y)} peptides, expected {expected}."
            )
        
        for other in datasets[1:]:
            if not np.array_equal(base.y, other.y):
                raise ValueError(
                    f"{split}/{feature}: feature blocks differ in row count or label order."
                )
            
        X = pd.concat([d.X for d in datasets], axis=1)
        if split != "train_beetle" and set(X.columns) != set(
            result["train_beetle"].X.columns
        ):
            raise ValueError(
                f"{split}/{feature}: feature headers differ from training."
            )
        if split != "train_beetle":
            X = X.loc[:, result["train_beetle"].X.columns]
        result[split] = Dataset(X, base.y)
        print(
            f"{split}/{feature}: {X.shape}, class counts {np.bincount(base.y).tolist()}"
        )
    return result

## Shared evaluation and exports

The notebook exports per-peptide scores, binary calls, original CSV row indices (zero-based), and any supplied identifiers.

In [30]:
METRICS = ['ACC', 'ROC_AUC', 'AP', 'MCC', 'F1', 'Sensitivity', 'Specificity']


def evaluate(dataset, probabilities, split, model, feature, seed):
    p = np.asarray(probabilities, dtype=float).reshape(-1)
    y = dataset.y

    if (p.shape != y.shape
        or not np.isfinite(p).all()
        or np.any((p < 0) | (p > 1))
    ):
        raise ValueError(
            f'{model}/{feature}/{split}: invalid positive-class probabilities.'
        )

    pred = (p >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    auc = roc_auc_score(y, p)

    if np.unique(pred).size == 1 or auc < 0.5:
        warnings.warn(
            f'{model}/{feature}/{split}: inspect score distributions '
            f'and label alignment (ROC-AUC={auc:.3f}).'
        )

    row = dict(
        dataset=split,
        model=model,
        feature_set=feature,
        seed=seed,
        threshold=THRESHOLD,
    )

    output = pd.DataFrame(
        dict(
            row,
            row_index=np.arange(len(y)),
            y_true=y,
            proba=p,
            y_pred=pred,
        )
    )
    output.to_csv(
        RUN_DIR / f'{model}_{feature}_{seed}_{split}_predictions.csv',
        index=False,
    )

    return dict(
        row,
        n=len(y),
        ACC=float(np.mean(y == pred)),
        ROC_AUC=auc,
        AP=average_precision_score(y, p),
        MCC=matthews_corrcoef(y, pred),
        F1=f1_score(y, pred, zero_division=0),
        Sensitivity=tp / (tp + fn),
        Specificity=tn / (tn + fp),
        TN=tn,
        FP=fp,
        FN=fn,
        TP=tp,
    )


def save_metrics(rows, prefix):
    if not rows:
        return

    frame = pd.DataFrame(rows)
    frame.to_csv(
        RUN_DIR / f'{prefix}_metrics_per_run.csv',
        index=False,
    )
    summary = frame.groupby(['dataset','model','feature_set'])[METRICS].agg(['count','mean','std'])
    summary.columns = ['_'.join(column) for column in summary.columns]
    summary.reset_index().to_csv(RUN_DIR / f'{prefix}_metrics_summary.csv', index=False)
    print(frame.round(4).to_string(index=False))

## Classical classifiers

An individual feature without documented settings uses the F1 reference settings as a fixed baseline, recorded in the run configuration. The scaler and the model are fitted on the complete training partition.

In [31]:
ML_PARAMETERS = {'F2': {'LR': {'solver': 'lbfgs', 'penalty': 'l2', 'C': 1},
        'RF': {'n_estimators': 200,
               'min_samples_split': 5,
               'min_samples_leaf': 2,
               'max_features': 'sqrt',
               'max_depth': 10},
        'ET': {'n_estimators': 500,
               'min_samples_split': 5,
               'min_samples_leaf': 2,
               'max_features': 'sqrt',
               'max_depth': 10},
        'XGB': {'subsample': 0.8,
                'n_estimators': 500,
                'max_depth': 10,
                'learning_rate': 0.01,
                'colsample_bytree': 0.8}},
 'F1': {'LR': {'solver': 'liblinear', 'penalty': 'l2', 'C': 0.1},
        'RF': {'n_estimators': 500,
               'min_samples_split': 10,
               'min_samples_leaf': 2,
               'max_features': 'sqrt',
               'max_depth': 10},
        'ET': {'n_estimators': 500,
               'min_samples_split': 2,
               'min_samples_leaf': 2,
               'max_features': 'sqrt',
               'max_depth': 10},
        'XGB': {'subsample': 0.8,
                'n_estimators': 500,
                'max_depth': 10,
                'learning_rate': 0.01,
                'colsample_bytree': 0.8}}}

In [33]:
def fit_classical(name, feature, data, seed):
    from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
    from sklearn.linear_model import LogisticRegression

    factory = {'LR': LogisticRegression, 'RF': RandomForestClassifier, 'ET': ExtraTreesClassifier}
    if name == 'XGB':
        from xgboost import XGBClassifier
        factory['XGB'] = XGBClassifier
    source = feature if feature in ML_PARAMETERS else 'F2'
    settings = dict(ML_PARAMETERS[source][name], random_state=seed)
    if name != 'LR':
        settings['n_jobs'] = N_JOBS
    if name == 'XGB':
        settings.update(objective='binary:logistic', eval_metric='logloss')
    train = data['train_beetle']
    scaler = StandardScaler().fit(train.X)
    classifier = factory[name](**settings).fit(scaler.transform(train.X), train.y)
    positive_column = list(classifier.classes_).index(1)
    (RUN_DIR / f'{name}_{feature}_{seed}_settings.json').write_text(
        json.dumps(dict(parameter_source=source, parameters=settings, scaler='StandardScaler'), indent=2))
    return [evaluate(data[s], classifier.predict_proba(scaler.transform(data[s].X))[:,positive_column],
                     s,name,feature,seed) for s in TESTS]

ml_rows = []
for feature, models in ML_JOBS.items():
    print(f"Fitting classical models for feature set {feature}...")
    data = load_feature(feature)
    for model in models:
        for seed in SEEDS:
            ml_rows.extend(fit_classical(model, feature, data, seed))
save_metrics(ml_rows, 'classical')

Fitting classical models for feature set PSTPP...
train_beetle/PSTPP: (119503, 23), class counts [56615, 62888]
val_beetle/PSTPP: (10000, 23), class counts [5000, 5000]
test_beetle/PSTPP: (10000, 23), class counts [5000, 5000]
    dataset model feature_set  seed  threshold     n    ACC  ROC_AUC     AP    MCC     F1  Sensitivity  Specificity   TN  FP  FN   TP
 val_beetle   XGB       PSTPP    42        0.5 10000 0.9458   0.9913 0.9918 0.8917 0.9454       0.9376       0.9540 4770 230 312 4688
test_beetle   XGB       PSTPP    42        0.5 10000 0.9497   0.9918 0.9923 0.8995 0.9493       0.9426       0.9568 4784 216 287 4713
 val_beetle    RF       PSTPP    42        0.5 10000 0.9453   0.9908 0.9913 0.8907 0.9449       0.9380       0.9526 4763 237 310 4690
test_beetle    RF       PSTPP    42        0.5 10000 0.9489   0.9911 0.9916 0.8979 0.9486       0.9426       0.9552 4776 224 287 4713
 val_beetle    LR       PSTPP    42        0.5 10000 0.9447   0.9904 0.9910 0.8895 0.9444       0.9386 

## Deep learning

The deep learning architectures will be rebuilt from their provided configurations with fresh weights and a width matching the selected features. CNN sees descriptors as a one-channel vector. Recurrent configurations receive one timestep containing the descriptor vector. The scaler and class weights are fitted on the stratified fitting subset. Validation uses the rest of the training partition for early stopping and learning rate reduction. The monitor defaults to validation loss.

In [ ]:
MODEL_FILES = {'CNN': 'model_cnn.json', 'FCNN': 'model_dnn.json', 'LSTM': 'model_lstm.json', 'GRU': 'model_gru.json', 'RNN': 'model_rnn.json'}
MODEL_CONFIGS = json.loads('{"CNN":{"config":{"name":"sequential_5","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"layers":[{"module":"keras.layers","class_name":"InputLayer","config":{"batch_shape":[null,48,1],"dtype":"float32","sparse":false,"ragged":false,"name":"input_layer_5","optional":false},"registered_name":null},{"module":"keras.layers","class_name":"Conv1D","config":{"name":"conv1d","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"filters":64,"kernel_size":[8],"strides":[1],"padding":"same","data_format":"channels_last","dilation_rate":[1],"groups":1,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"bias_constraint":null},"registered_name":null,"build_config":{"input_shape":[null,48,1]}},{"module":"keras.layers","class_name":"MaxPooling1D","config":{"name":"max_pooling1d","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"pool_size":[2],"padding":"valid","strides":[2],"data_format":"channels_last"},"registered_name":null},{"module":"keras.layers","class_name":"BatchNormalization","config":{"name":"batch_normalization","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"axis":-1,"momentum":0.99,"epsilon":0.001,"center":true,"scale":true,"beta_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"gamma_initializer":{"module":"keras.initializers","class_name":"Ones","config":{},"registered_name":null},"moving_mean_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"moving_variance_initializer":{"module":"keras.initializers","class_name":"Ones","config":{},"registered_name":null},"beta_regularizer":null,"gamma_regularizer":null,"beta_constraint":null,"gamma_constraint":null,"synchronized":false},"registered_name":null,"build_config":{"input_shape":[null,24,64]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_6","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.4,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Conv1D","config":{"name":"conv1d_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"filters":32,"kernel_size":[8],"strides":[1],"padding":"same","data_format":"channels_last","dilation_rate":[1],"groups":1,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"bias_constraint":null},"registered_name":null,"build_config":{"input_shape":[null,24,64]}},{"module":"keras.layers","class_name":"MaxPooling1D","config":{"name":"max_pooling1d_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"pool_size":[2],"padding":"valid","strides":[2],"data_format":"channels_last"},"registered_name":null},{"module":"keras.layers","class_name":"BatchNormalization","config":{"name":"batch_normalization_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"axis":-1,"momentum":0.99,"epsilon":0.001,"center":true,"scale":true,"beta_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"gamma_initializer":{"module":"keras.initializers","class_name":"Ones","config":{},"registered_name":null},"moving_mean_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"moving_variance_initializer":{"module":"keras.initializers","class_name":"Ones","config":{},"registered_name":null},"beta_regularizer":null,"gamma_regularizer":null,"beta_constraint":null,"gamma_constraint":null,"synchronized":false},"registered_name":null,"build_config":{"input_shape":[null,12,32]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_7","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.2,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Flatten","config":{"name":"flatten_3","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"data_format":"channels_last"},"registered_name":null,"build_config":{"input_shape":[null,12,32]}},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_11","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":192,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,384]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_8","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.2,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_12","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":1,"activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":null,"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,192]}}],"build_input_shape":[null,48,1]},"compile_config":{"optimizer":{"module":"keras.optimizers","class_name":"RMSprop","config":{"name":"rmsprop","learning_rate":0.0010000000474974513,"weight_decay":null,"clipnorm":null,"global_clipnorm":null,"clipvalue":null,"use_ema":false,"ema_momentum":0.99,"ema_overwrite_frequency":null,"loss_scale_factor":null,"gradient_accumulation_steps":null,"rho":0.9,"momentum":0.0,"epsilon":1e-07,"centered":false},"registered_name":null},"loss":"binary_crossentropy","loss_weights":null,"metrics":["accuracy"],"weighted_metrics":null,"run_eagerly":false,"steps_per_execution":1,"jit_compile":true}},"FCNN":{"config":{"name":"sequential","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"layers":[{"module":"keras.layers","class_name":"InputLayer","config":{"batch_shape":[null,48],"dtype":"float32","sparse":false,"ragged":false,"name":"input_layer","optional":false},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":352,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,48]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.5,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":352,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,352]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.5,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_2","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":1,"activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":null,"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,352]}}],"build_input_shape":[null,48]},"compile_config":{"optimizer":{"module":"keras.optimizers","class_name":"RMSprop","config":{"name":"rmsprop","learning_rate":0.0010000000474974513,"weight_decay":null,"clipnorm":null,"global_clipnorm":null,"clipvalue":null,"use_ema":false,"ema_momentum":0.99,"ema_overwrite_frequency":null,"loss_scale_factor":null,"gradient_accumulation_steps":null,"rho":0.9,"momentum":0.0,"epsilon":1e-07,"centered":false},"registered_name":null},"loss":"binary_crossentropy","loss_weights":null,"metrics":["accuracy"],"weighted_metrics":null,"run_eagerly":false,"steps_per_execution":1,"jit_compile":true}},"LSTM":{"config":{"name":"sequential_4","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"layers":[{"module":"keras.layers","class_name":"InputLayer","config":{"batch_shape":[null,1,48],"dtype":"float32","sparse":false,"ragged":false,"name":"input_layer_4","optional":false},"registered_name":null},{"module":"keras.layers","class_name":"LSTM","config":{"name":"lstm","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"return_sequences":true,"return_state":false,"go_backwards":false,"stateful":false,"unroll":false,"zero_output_for_mask":false,"units":96,"activation":"relu","recurrent_activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"recurrent_initializer":{"module":"keras.initializers","class_name":"Orthogonal","config":{"seed":null,"gain":1.0},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"unit_forget_bias":true,"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.0001},"registered_name":null},"recurrent_regularizer":null,"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"recurrent_constraint":null,"bias_constraint":null,"dropout":0.1,"recurrent_dropout":0.0,"seed":null},"registered_name":null,"build_config":{"input_shape":[null,1,48]}},{"module":"keras.layers","class_name":"LSTM","config":{"name":"lstm_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"return_sequences":false,"return_state":false,"go_backwards":false,"stateful":false,"unroll":false,"zero_output_for_mask":false,"units":96,"activation":"relu","recurrent_activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"recurrent_initializer":{"module":"keras.initializers","class_name":"Orthogonal","config":{"seed":null,"gain":1.0},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"unit_forget_bias":true,"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.0001},"registered_name":null},"recurrent_regularizer":null,"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"recurrent_constraint":null,"bias_constraint":null,"dropout":0.5,"recurrent_dropout":0.0,"seed":null},"registered_name":null,"build_config":{"input_shape":[null,1,96]}},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_9","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":32,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.0001},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,96]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_5","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.2,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_10","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":1,"activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":null,"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,32]}}],"build_input_shape":[null,1,48]},"compile_config":{"optimizer":{"module":"keras.optimizers","class_name":"RMSprop","config":{"name":"rmsprop","learning_rate":0.0010000000474974513,"weight_decay":null,"clipnorm":null,"global_clipnorm":null,"clipvalue":null,"use_ema":false,"ema_momentum":0.99,"ema_overwrite_frequency":null,"loss_scale_factor":null,"gradient_accumulation_steps":null,"rho":0.9,"momentum":0.0,"epsilon":1e-07,"centered":false},"registered_name":null},"loss":"binary_crossentropy","loss_weights":null,"metrics":["accuracy"],"weighted_metrics":null,"run_eagerly":false,"steps_per_execution":1,"jit_compile":true}},"GRU":{"config":{"name":"sequential_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"layers":[{"module":"keras.layers","class_name":"InputLayer","config":{"batch_shape":[null,1,48],"dtype":"float32","sparse":false,"ragged":false,"name":"input_layer_1","optional":false},"registered_name":null},{"module":"keras.layers","class_name":"GRU","config":{"name":"gru","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"return_sequences":false,"return_state":false,"go_backwards":false,"stateful":false,"unroll":false,"zero_output_for_mask":false,"units":64,"activation":"relu","recurrent_activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"recurrent_initializer":{"module":"keras.initializers","class_name":"Orthogonal","config":{"seed":null,"gain":1.0},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"recurrent_regularizer":null,"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"recurrent_constraint":null,"bias_constraint":null,"dropout":0.1,"recurrent_dropout":0.0,"reset_after":true,"seed":null},"registered_name":null,"build_config":{"input_shape":[null,1,48]}},{"module":"keras.layers","class_name":"Flatten","config":{"name":"flatten","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"data_format":"channels_last"},"registered_name":null,"build_config":{"input_shape":[null,64]}},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_3","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":32,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,64]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_2","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.30000000000000004,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_4","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":1,"activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":null,"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,32]}}],"build_input_shape":[null,1,48]},"compile_config":{"optimizer":{"module":"keras.optimizers","class_name":"RMSprop","config":{"name":"rmsprop","learning_rate":0.0010000000474974513,"weight_decay":null,"clipnorm":null,"global_clipnorm":null,"clipvalue":null,"use_ema":false,"ema_momentum":0.99,"ema_overwrite_frequency":null,"loss_scale_factor":null,"gradient_accumulation_steps":null,"rho":0.9,"momentum":0.0,"epsilon":1e-07,"centered":false},"registered_name":null},"loss":"binary_crossentropy","loss_weights":null,"metrics":["accuracy"],"weighted_metrics":null,"run_eagerly":false,"steps_per_execution":1,"jit_compile":true}},"RNN":{"config":{"name":"sequential_3","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"layers":[{"module":"keras.layers","class_name":"InputLayer","config":{"batch_shape":[null,1,48],"dtype":"float32","sparse":false,"ragged":false,"name":"input_layer_3","optional":false},"registered_name":null},{"module":"keras.layers","class_name":"GRU","config":{"name":"gru_2","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"return_sequences":false,"return_state":false,"go_backwards":false,"stateful":false,"unroll":false,"zero_output_for_mask":false,"units":16,"activation":"relu","recurrent_activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"recurrent_initializer":{"module":"keras.initializers","class_name":"Orthogonal","config":{"seed":null,"gain":1.0},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.0001},"registered_name":null},"recurrent_regularizer":null,"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"recurrent_constraint":null,"bias_constraint":null,"dropout":0.1,"recurrent_dropout":0.0,"reset_after":true,"seed":null},"registered_name":null,"build_config":{"input_shape":[null,1,48]}},{"module":"keras.layers","class_name":"Flatten","config":{"name":"flatten_2","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"data_format":"channels_last"},"registered_name":null,"build_config":{"input_shape":[null,16]}},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_7","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":320,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.0001},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,16]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_4","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.2,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_8","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":1,"activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":null,"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,320]}}],"build_input_shape":[null,1,48]},"compile_config":{"optimizer":{"module":"keras.optimizers","class_name":"RMSprop","config":{"name":"rmsprop","learning_rate":0.009999999776482582,"weight_decay":null,"clipnorm":null,"global_clipnorm":null,"clipvalue":null,"use_ema":false,"ema_momentum":0.99,"ema_overwrite_frequency":null,"loss_scale_factor":null,"gradient_accumulation_steps":null,"rho":0.9,"momentum":0.0,"epsilon":1e-07,"centered":false},"registered_name":null},"loss":"binary_crossentropy","loss_weights":null,"metrics":["accuracy"],"weighted_metrics":null,"run_eagerly":false,"steps_per_execution":1,"jit_compile":true}}}')

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
def build_deep_model(name, width):
    spec = MODEL_CONFIGS[name] if MODEL_JSON_DIR is None else json.loads(
        (Path(MODEL_JSON_DIR) / MODEL_FILES[name]).read_text())
    layers = spec['config']['layers']
    input_config = layers[0]['config']
    shape = tuple((input_config.get('batch_shape') or input_config['batch_input_shape'])[1:])
    if len(shape) == 1:
        input_shape = (width,)
    elif shape[0] == 1:
        input_shape = (1,width)
    elif shape[-1] == 1:
        input_shape = (width,1)
    else:
        raise ValueError(f'Unsupported JSON input shape: {shape}')
    model = tf.keras.Sequential([tf.keras.Input(shape=input_shape, dtype='float32')], name=name)
    for original in layers[1:]:
        layer = copy.deepcopy(original)
        layer.pop('build_config', None)
        model.add(tf.keras.layers.deserialize(layer))
    if model.output_shape != (None,1) or getattr(model.layers[-1].activation,'__name__','') != 'sigmoid':
        raise ValueError(f'{name}: expected one sigmoid output per peptide.')
    if name == 'RNN_JSON_GRU' and not any(isinstance(layer, tf.keras.layers.GRU) for layer in model.layers):
        raise ValueError('Register a corrected RNN JSON under its own model name before using it.')
    optimizer = tf.keras.optimizers.deserialize(spec['compile_config']['optimizer'])
    model.compile(optimizer=optimizer, loss=spec['compile_config']['loss'],
                  metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy')])
    return model, input_shape

In [ ]:
class SelectiveProgbarLogger(tf.keras.callbacks.ProgbarLogger):
    def __init__(self, verbose, epoch_interval, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.default_verbose = verbose
        self.epoch_interval = epoch_interval
    
    def on_epoch_begin(self, epoch, *args, **kwargs):
        self.verbose = (
            0 
                if epoch % self.epoch_interval != 0 
                else self.default_verbose
        )
        super().on_epoch_begin(epoch, *args, **kwargs)
        
def fit_deep(name, feature, data, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    train = data['train_beetle']
    val = data['val_beetle']
    fit_ids = np.arange(len(train.y))
    val_ids = np.arange(len(val.y))
    scaler = StandardScaler().fit(train.X.iloc[fit_ids])
    model, shape = build_deep_model(name, train.X.shape[1])

    def transform(frame):
        return scaler.transform(frame).astype('float32', copy=False).reshape((-1,*shape))

    class_weights = compute_class_weight('balanced', classes=np.array([0,1]), y=train.y[fit_ids])
    history = model.fit(
        transform(train.X.iloc[fit_ids]), train.y[fit_ids],
        validation_data=(transform(val.X.iloc[val_ids]), val.y[val_ids]),
        batch_size=BATCH_SIZE, epochs=MAX_EPOCHS, shuffle=True,
        class_weight=dict(enumerate(class_weights.tolist())),
        callbacks=[
            SelectiveProgbarLogger(verbose=0, epoch_interval=10),
            tf.keras.callbacks.EarlyStopping(monitor=MONITOR, patience=PATIENCE, mode=MODE, verbose=1, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor=MONITOR, mode=MODE, factor=0.5, patience=10, min_lr=1e-6, verbose=1),
            tf.keras.callbacks.ModelCheckpoint(filepath=RUN_DIR / f'{name}_{feature}_{seed}.keras', verbose=1, monitor=MONITOR, mode=MODE, save_best_only=True,
            ),
        ], verbose=0,
    )
    prefix = RUN_DIR / f'{name}_{feature}_{seed}'
    pd.DataFrame(history.history).rename_axis('epoch_zero_based').to_csv(f'{prefix}_history.csv')
    Path(f'{prefix}_settings.json').write_text(json.dumps(dict(
        input_shape=shape, scaler='StandardScaler', seed=seed,
        validation_fraction=VAL_FRACTION, monitor=MONITOR, patience=PATIENCE,
        max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, class_weights=class_weights.tolist(),
        architecture=json.loads(model.to_json())), indent=2))
    return [evaluate(data[s], model.predict(transform(data[s].X), batch_size=256, verbose=0),
                     s,name,feature,seed) for s in TESTS]

dl_rows = []
if DL_JOBS:
    for feature, models in DL_JOBS.items():
        data = load_feature(feature)
        for model in models:
            for seed in SEEDS:
                dl_rows.extend(fit_deep(model, feature, data, seed))
save_metrics(dl_rows, 'deep_learning')

## Reporting the results
Use `*_metrics_per_run.csv` for individual fits and `*_metrics_summary.csv` for repeated-run results. With one seed, SD is undefined and exported blank. AP denotes average precision. 